# Math & Neural Network Basics — Hands-On Notebook

Companion to: [Math Prerequisites](../docs/00_prerequisites/00_math_prerequisites.md), [Perceptron & FFN](../docs/00_prerequisites/01_perceptron_and_ffn.md), [Activation Functions](../docs/00_prerequisites/02_activation_functions.md)

**What you'll build:**
- Vectors, matrices, and dot products from scratch
- A single neuron, then a 2-layer MLP with NumPy
- The same MLP with PyTorch
- Visualize every activation function

## Section 1: Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

np.random.seed(42)
torch.manual_seed(42)

## Section 2: Vectors and Matrices

If we write **x ∈ ℝᵈ**, we mean a list of **d** real numbers — like a shopping list with **d** items on it. Position 1 might be “apples,” position 2 “milk,” and so on; in math we just store numbers in a fixed order. A **vector** is that ordered list; a **matrix** is a grid of numbers, useful for rotating, scaling, or mixing many lists at once.

The **dot product** combines two same-length vectors into one number (a measure of alignment). **Matrix–vector multiply** applies a linear transformation: each row of the matrix takes a dot product with the vector.

In [ ]:
x = np.array([1.0, 2.0, 3.0])
y = np.array([0.5, -1.0, 2.0])
print("x =", x)
print("y =", y)

dot_manual = sum(x[i] * y[i] for i in range(len(x)))
print("Dot product (manual):", dot_manual)
print("np.dot(x, y):", np.dot(x, y))

In [ ]:
W = np.array([[1.0, 0.0, 2.0], [0.0, 1.0, -1.0]])
print("W shape:", W.shape)
print("W @ x (matrix-vector):", W @ x)

row0 = np.dot(W[0], x)
row1 = np.dot(W[1], x)
print("Row 0 dot x:", row0)
print("Row 1 dot x:", row1)
print("Matches W @ x:", np.allclose(W @ x, np.array([row0, row1])))

## Section 3: Build a Single Neuron from Scratch

A **neuron** takes a vector of inputs, multiplies them by **weights**, adds a **bias**, then passes the result through an **activation** (nonlinear) function to produce its output.

In [ ]:
def sigmoid(z):
    z = np.clip(z, -500.0, 500.0)
    return 1.0 / (1.0 + np.exp(-z))


def neuron(x, w, b):
    z = np.dot(w, x) + b
    return sigmoid(z), z


x_demo = np.array([1.0, 2.0, 3.0])
w_demo = np.array([0.5, -0.3, 0.8])
b_demo = 0.1
out, z = neuron(x_demo, w_demo, b_demo)
print("z (pre-activation):", z)
print("sigmoid(z) (output):", out)

## Section 4: Build a 2-Layer MLP from Scratch

We **stack** neurons into **layers**: each layer computes a linear map followed by an activation. The output of one layer feeds the next, so we can represent rich patterns — input (3) → hidden (4) with **ReLU**, then hidden → output (2) with **sigmoid**.

In [ ]:
def relu(z):
    return np.maximum(0.0, z)


def mlp_numpy(x, W1, b1, W2, b2):
    z1 = W1 @ x + b1
    h = relu(z1)
    z2 = W2 @ h + b2
    out = sigmoid(z2)
    return z1, h, z2, out


rng = np.random.default_rng(42)
W1 = rng.normal(0, 0.5, size=(4, 3))
b1 = rng.normal(0, 0.1, size=(4,))
W2 = rng.normal(0, 0.5, size=(2, 4))
b2 = rng.normal(0, 0.1, size=(2,))

x_in = np.array([0.5, -0.2, 1.0])
z1, h, z2, out = mlp_numpy(x_in, W1, b1, W2, b2)
print("z1 (hidden pre-activation):", z1)
print("h (hidden after ReLU):", h)
print("z2 (logits):", z2)
print("output (sigmoid):", out)

## Section 5: The Same MLP with PyTorch

PyTorch performs the **same linear algebra**, but also builds a **computation graph** so **gradients** can be computed automatically with **autograd** — essential for training.

In [ ]:
model = nn.Sequential(
    nn.Linear(3, 4),
    nn.ReLU(),
    nn.Linear(4, 2),
    nn.Sigmoid(),
)

with torch.no_grad():
    model[0].weight.copy_(torch.from_numpy(W1.astype(np.float32)))
    model[0].bias.copy_(torch.from_numpy(b1.astype(np.float32)))
    model[2].weight.copy_(torch.from_numpy(W2.astype(np.float32)))
    model[2].bias.copy_(torch.from_numpy(b2.astype(np.float32)))

x_t = torch.tensor(x_in, dtype=torch.float32, requires_grad=True)
y_torch = model(x_t)
print("PyTorch output:", y_torch.detach().numpy())
print("NumPy output:  ", out)
print("Max abs diff:", np.max(np.abs(y_torch.detach().numpy() - out)))

loss = y_torch.sum()
loss.backward()
print("Gradient w.r.t. input x (sample):", x_t.grad.numpy())

## Section 6: Activation Function Zoo

**Why nonlinearities?** If we only stacked linear layers, the whole network would collapse to a **single** linear map: depth would not add new expressive power. Activations let us approximate curved boundaries and complex functions.

Below: each panel shows an activation **f(x)** and its derivative **f′(x)** (via PyTorch autograd for smooth, consistent curves).

In [ ]:
x_np = np.linspace(-5, 5, 200)
activations = [
    ("Sigmoid", lambda t: torch.sigmoid(t)),
    ("Tanh", torch.tanh),
    ("ReLU", F.relu),
    ("LeakyReLU (α=0.01)", lambda t: F.leaky_relu(t, negative_slope=0.01)),
    ("GELU", F.gelu),
    ("SiLU / Swish", F.silu),
]

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, (name, fn) in zip(axes.flat, activations):
    x_t = torch.tensor(x_np, dtype=torch.float32, requires_grad=True)
    y = fn(x_t)
    y.sum().backward()
    ax.plot(x_np, y.detach().numpy(), label="f(x)")
    ax.plot(x_np, x_t.grad.numpy(), label="f'(x)")
    ax.set_title(name)
    ax.set_xlabel("x")
    ax.set_ylabel("value")
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.axvline(0, color="gray", linewidth=0.5)
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)
plt.suptitle("Activations and derivatives")
plt.tight_layout()
plt.show()

### Softmax on a sample vector

**Softmax** turns three logits into three positive probabilities that sum to 1 — useful for multi-class outputs.

In [ ]:
logits = np.array([2.0, 1.0, 0.1], dtype=np.float64)
stable = logits - logits.max()
probs = np.exp(stable) / np.exp(stable).sum()
print("Softmax probabilities:", probs)

labels = ["class 0", "class 1", "class 2"]
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(labels, probs, color=["#4C72B0", "#55A868", "#C44E52"])
ax.set_ylabel("probability")
ax.set_title("Softmax for logits [2.0, 1.0, 0.1]")
ax.set_ylim(0, 1)
for i, v in enumerate(probs):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center")
plt.tight_layout()
plt.show()

## Section 7: Comparing Activations with PyTorch (XOR)

We train a small two-layer network on the **XOR** problem (not linearly separable) and compare different **hidden** activations.

In [ ]:
X_xor = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = torch.tensor([[0.0], [1.0], [1.0], [0.0]])


def build_xor_net(hidden_act):
    return nn.Sequential(nn.Linear(2, 8), hidden_act, nn.Linear(8, 1), nn.Sigmoid())


def train_xor(activation_ctor, epochs=4000, lr=0.05):
    torch.manual_seed(42)
    model = build_xor_net(activation_ctor())
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()
    losses = []
    for _ in range(epochs):
        opt.zero_grad()
        pred = model(X_xor)
        loss = loss_fn(pred, y_xor)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses


runs = {
    "ReLU": lambda: nn.ReLU(),
    "Sigmoid": lambda: nn.Sigmoid(),
    "Tanh": lambda: nn.Tanh(),
    "GELU": lambda: nn.GELU(),
}

history = {name: train_xor(ctor) for name, ctor in runs.items()}

fig, ax = plt.subplots(figsize=(8, 5))
for name, losses in history.items():
    ax.plot(losses, label=name, alpha=0.9)
ax.set_yscale("log")
ax.set_xlabel("epoch")
ax.set_ylabel("BCE loss (log scale)")
ax.set_title("XOR training loss by hidden activation")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

for name, losses in history.items():
    print(f"{name}: final loss = {losses[-1]:.6f}")

**Notice how ReLU and GELU converge faster** — this is why modern networks prefer them.

## Section 8: Key Takeaways

- **Vectors and matrices** encode data and linear transformations; dot products and matrix–vector products are the building blocks of neural layers.
- A **neuron** is: linear combination of inputs + **bias**, then an **activation**.
- Stacking layers yields an **MLP**; **ReLU** (or **GELU**) in hidden layers is standard today; **sigmoid** often appears at the output for probabilities.
- **Nonlinear activations** are what make depth useful; without them, multiple layers equal one linear map.
- **PyTorch** matches NumPy’s forward math but adds **autograd** for training.
- On small hard problems (like XOR), **ReLU** and **GELU** typically optimize faster than **sigmoid**/**tanh** hidden units — one reason modern networks prefer them.